# Region splitting and dataset indexing flow

This notebook confirms two related mechanisms: how `SplitRegions.from_ratios` partitions a
global crop into disjoint train, validation and test regions along azimuth, and how
`MultiRegionDataset` concatenates per-region `PatchDataset` parts so that a flat global index
maps to the correct region and local patch via the `__getitem__` flow.

Modules exercised:

- `tools.regions.SplitRegions`
- `tools.regions.CropRegion`
- `pipelines.dataset_pipeline.datasets.PatchDataset`
- `pipelines.dataset_pipeline.datasets.MultiRegionDataset`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Split a global crop by ratio

We define a global crop and split it 70/15/15 along azimuth. The regions must be contiguous,
disjoint and cover the full azimuth extent.


In [ ]:
from tools.regions                        import SplitRegions, CropRegion
from tools.logger                         import Logger
from configuration.dataset_config         import InputConfig, OutputConfig
from configuration.representation         import Representation
from pipelines.dataset_pipeline.spatial   import Patcher
from pipelines.dataset_pipeline.datasets  import PatchDataset, MultiRegionDataset

logger      = Logger(log_dir="/tmp/dataset_nb_logs", name="nb10", level="WARNING")
global_crop = CropRegion(azimuth_start=0, azimuth_end=200, range_start=0, range_end=120)
splits      = SplitRegions.from_ratios(global_crop, train_ratio=0.70, val_ratio=0.15)

for name, region in splits.items():
    print(f"{name:6s}: azimuth [{region.azimuth_start:4d}, {region.azimuth_end:4d})  size={region.azimuth_size}")


In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(7, 4))
colors  = {"train": "C0", "val": "C1", "test": "C2"}
for name, region in splits.items():
    rect = mpatches.Rectangle((region.range_start, region.azimuth_start), region.range_size, region.azimuth_size,
                              facecolor=colors[name], edgecolor="k", alpha=0.6, label=name)
    ax.add_patch(rect)
ax.set_xlim(0, global_crop.range_end); ax.set_ylim(global_crop.azimuth_end, 0)
ax.set_xlabel("range"); ax.set_ylabel("azimuth")
ax.set_title("azimuth-wise train / val / test split")
ax.legend()
plt.tight_layout(); plt.show()



## Build per-region datasets

We make a multi-region training split by supplying two disjoint train regions, each backed by
its own synthetic stack, then wrap them in a `MultiRegionDataset`.


In [ ]:
def build_part(az_size, rg_size, seed):
    rng   = np.random.default_rng(seed)
    stack = (rng.standard_normal((1, az_size, rg_size)) + 1j * rng.standard_normal((1, az_size, rg_size))).astype(np.complex64)
    gt    = rng.random((3, az_size, rg_size)).astype(np.float32)
    patcher = Patcher.build(spatial_size=(az_size, rg_size), patch_size=(32, 32), stride=32)
    cfg     = InputConfig(use_primary=True, primary_representation=Representation.MAG_ONLY,
                          use_secondaries=False, use_interferograms=False, use_dem=False)
    return PatchDataset(inputs=stack, gt_parameters=gt, grid=patcher, input_config=cfg,
                        output_config=OutputConfig(), split_name="train",
                        n_secondaries=0, n_interferograms=0, n_gaussians=1)

part_a = build_part(64, 96, seed=1)
part_b = build_part(96, 96, seed=2)
multi  = MultiRegionDataset([part_a, part_b])

print("part A patches:", len(part_a))
print("part B patches:", len(part_b))
print("multi  patches:", len(multi), " offsets:", multi.offsets)



## Global index routing

A flat index below the first offset addresses part A; at or above it addresses part B. We
confirm by comparing items pulled through the global index against items pulled directly from
each part.


In [ ]:
boundary = multi.offsets[1]
checks   = [0, boundary - 1, boundary, len(multi) - 1]
for g in checks:
    xg, _ = multi[g]
    if g < boundary:
        xd, _ = part_a[g]
        src   = "A"
    else:
        xd, _ = part_b[g - boundary]
        src   = "B"
    print(f"global {g:3d} -> part {src} local {g if g < boundary else g - boundary:3d}  match={np.allclose(xg, xd)}")


In [ ]:
fig, axes = plt.subplots(1, len(checks), figsize=(2.4 * len(checks), 2.6))
for ax, g in zip(axes, checks):
    xg, _ = multi[g]
    ax.imshow(xg[0], cmap="magma")
    ax.set_title(f"global idx {g}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## Expected visual outcome

The split panel should show three stacked, non-overlapping azimuth bands covering the whole
crop with sizes near 140/30/30 lines. The total patch count of the multi-region dataset should
equal the sum of its parts, and every probed global index should route to the correct part and
local index with `match=True`, confirming the `bisect`-based `__getitem__` routing.